# Aquarium Animal Census — RT-DETR fine-tune on Kaggle

Fine-tunes RT-DETR-L to detect seven aquarium animals — **fish, jellyfish,
penguin, puffin, shark, starfish, stingray** (none of them a COCO class) — on
Roboflow's *Aquarium Combined* photographs, evaluates it on a held-out test
split, mines failure cases with measured evidence, checks the census reasoning
layer end to end, and packages the checkpoint for the FastAPI service in the repo.

**Notebook settings (right-hand panel):**

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** (recommended). P100 also works: Kaggle's current torch build has dropped sm_60 support, so cell 1 detects that and installs a compatible torch first. |
| Internet | **ON** — needed for pip, `kagglehub`, and the `rtdetr-l.pt` base weights |
| Datasets to attach | **None.** The dataset is downloaded by code below via `kagglehub`. |

Run through **Save Version → Save & Run All** so a browser disconnect does
not kill the run. Everything is written under `/kaggle/working`. Expect about
45 minutes end to end on 2 × T4.

## 1. Environment

Installs the pinned dependencies, then checks that the preinstalled torch was
compiled for this GPU. Kaggle's torch 2.10 + CUDA 12.8 image no longer includes
sm_60 kernels, so on a P100 the model fails with `CUDA error: no kernel image
is available`. If that mismatch is detected, a torch build that still supports
the card is installed **before** torch is imported into this kernel.

In [ ]:
import os, sys, subprocess, json
from pathlib import Path
os.makedirs("/kaggle/working/repo", exist_ok=True)
os.chdir("/kaggle/working/repo")

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("ultralytics==8.3.40", "kagglehub")   # no numpy/opencv pins: Kaggle ships numpy 2 and cv2 already
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

# Probe in a subprocess so torch is not yet imported here if it has to be replaced.
probe = subprocess.run([sys.executable, "-c",
    "import torch; cap = torch.cuda.get_device_capability(); "
    "print('sm_%d%d' % cap); print(' '.join(torch.cuda.get_arch_list())); print(torch.__version__)"],
    capture_output=True, text=True)
gpu_sm, arch_list, torch_ver = (probe.stdout.strip().split("\n") + ["", "", ""])[:3]
print(f"gpu {gpu_sm} | torch {torch_ver} compiled for: {arch_list}")

if gpu_sm and gpu_sm not in arch_list.split():
    print(f"{gpu_sm} is not supported by the preinstalled torch; installing torch 2.6.0 (cu126), ~2.5 GB ...")
    pip("--force-reinstall", "--no-deps",
        "torch==2.6.0", "torchvision==0.21.0",
        "--index-url", "https://download.pytorch.org/whl/cu126")
    pip("ultralytics==8.3.40")   # re-satisfy deps the --no-deps install skipped

import torch, ultralytics
sm = "sm_%d%d" % torch.cuda.get_device_capability()
assert sm in torch.cuda.get_arch_list(), f"{sm} still unsupported by torch {torch.__version__}: {torch.cuda.get_arch_list()}"
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| gpu", torch.cuda.get_device_name(0),
      "| ultralytics", ultralytics.__version__)

## 2. Run configuration

Edit here, nowhere else. Set `DRY_RUN = True` the first time to measure
seconds per epoch, then multiply it out before committing to `EPOCHS`.
RT-DETR is attention heavy: **batch 8 per GPU at 640 px is the largest that
fits on a 16 GB card**. With two GPUs the batch is doubled and Ultralytics
trains with DDP across both; the per-card load stays at 8. The dataset is
small (about 450 training images, ~28 steps per epoch), so the budget goes
into epochs: 80 epochs with early stopping at 25 epochs of no improvement.

In [ ]:
DRY_RUN   = False   # True -> 2 epochs into runs/dryrun, then stop
EPOCHS    = 80
PATIENCE  = 25
N_GPU     = torch.cuda.device_count()
for i in range(N_GPU):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}  {torch.cuda.get_device_properties(i).total_memory/2**30:.1f} GB")
if N_GPU < 2:
    print(f"WARNING: only {N_GPU} GPU visible. Training will still run, ~2x slower. Pick 'GPU T4 x2' next time.")
DEVICE    = ",".join(str(i) for i in range(N_GPU)) or "cpu"   # "0,1" -> Ultralytics relaunches under DDP across both cards
BATCH     = 8 * max(N_GPU, 1)                                 # 8 per GPU is the 16 GB ceiling for RT-DETR-L at 640
WORKERS   = os.cpu_count()                           # Kaggle gives 4 cores; the dataloader is the bottleneck otherwise
CACHE     = "ram"                                    # 638 decoded images at 640 px is well under 1 GB
IMGSZ     = 640
LR0       = 1e-4
SEED      = 0

# Public Kaggle mirrors of Roboflow's "Aquarium Combined" (CC BY 4.0). The first
# that downloads is used; scripts/prepare_data.py reads any YOLO layout.
DATASET_CANDIDATES = ["slavkoprytula/aquarium-data-cots",
                      "sharansmenon/aquarium-dataset",
                      "sovitrath/aquarium-data"]

WORK  = "/kaggle/working"
DATA  = f"{WORK}/data/aquarium"
RUNS  = f"{WORK}/runs"
ARTS  = f"{WORK}/artifacts"
RUN_NAME = "dryrun" if DRY_RUN else "rtdetr_aquarium"
RUN_EPOCHS = 2 if DRY_RUN else EPOCHS
BEST = f"{RUNS}/{RUN_NAME}/weights/best.pt"
os.makedirs(ARTS, exist_ok=True)
print(json.dumps({k: v for k, v in globals().items() if k.isupper() and not k.startswith("_")}, indent=2, default=str))

## 3. Repository code

The repo is embedded below so the notebook is self-contained. These cells are
generated from the real source files by `notebooks/build_kaggle_notebook.py`;
do not edit them here.

In [ ]:
from pathlib import Path
Path("/kaggle/working/repo/scripts/__init__.py").parent.mkdir(parents=True, exist_ok=True)
Path("/kaggle/working/repo/scripts/__init__.py").touch()
print("created empty", "/kaggle/working/repo/scripts/__init__.py")

In [ ]:
%%writefile /kaggle/working/repo/scripts/prepare_data.py
"""Convert the Kaggle mirror of Roboflow's "Aquarium Combined" dataset into a
clean YOLO dataset with a deterministic, image-level train/val/test split.

Source: https://www.kaggle.com/datasets/slavkoprytula/aquarium-data-cots
        (Roboflow "Aquarium Combined", CC BY 4.0: 638 photographs taken at the
        Henry Doorly Zoo, Omaha and the National Aquarium, Baltimore)

The mirror ships YOLO txt labels in Roboflow's train/valid/test folders. This
script does not trust that layout: it walks the whole tree for images that
have a label file, removes byte-identical duplicates (Roboflow exports can
place the same frame in two splits under different names), maps class ids to
this repo's fixed class order by NAME, and re-splits everything with a hash
of the file stem so the split never moves between reruns.

Usage:
    python scripts/prepare_data.py \
        --root /kaggle/input/aquarium-data-cots \
        --out  /kaggle/working/data/aquarium
"""
import argparse
import hashlib
import json
import re
import shutil
from collections import Counter
from pathlib import Path

# Fixed class order. The index is baked into the weights, so never reorder it.
CLASSES = ["fish", "jellyfish", "penguin", "puffin", "shark", "starfish", "stingray"]
CLASS_TO_ID = {name: i for i, name in enumerate(CLASSES)}

# How the source may spell each class. Everything else is dropped, never guessed.
ALIASES = {
    "fish": "fish",
    "jellyfish": "jellyfish", "jelly fish": "jellyfish", "jelly-fish": "jellyfish",
    "penguin": "penguin",
    "puffin": "puffin",
    "shark": "shark",
    "starfish": "starfish", "star fish": "starfish", "star-fish": "starfish", "sea star": "starfish",
    "stingray": "stingray", "sting ray": "stingray", "sting-ray": "stingray", "ray": "stingray",
}

IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


def read_source_names(root: Path):
    """Class names in the source's own id order, from any yaml under root.

    Returns a list (index -> name) or None if no yaml declares `names`.
    """
    for yaml_path in sorted(list(root.rglob("*.yaml")) + list(root.rglob("*.yml"))):
        text = yaml_path.read_text(encoding="utf-8", errors="replace")
        if "names" not in text:
            continue
        try:
            import yaml
            doc = yaml.safe_load(text) or {}
        except Exception:
            continue
        names = doc.get("names")
        if isinstance(names, dict):
            return [str(names[k]) for k in sorted(names, key=int)]
        if isinstance(names, list):
            return [str(n) for n in names]
    return None


def find_pairs(root: Path):
    """Yield (image_path, label_path_or_None, source_split) for every image.

    Handles the two YOLO layouts in the wild -- `<split>/images/*.jpg` with
    `<split>/labels/*.txt`, and `images/<split>/*.jpg` with
    `labels/<split>/*.txt` -- by mirroring each image's path below `images/`
    into the sibling `labels/`. If no images/ + labels/ pair exists at all,
    falls back to any image with a same-stem .txt beside it.
    """
    seen = set()
    for images_dir in sorted(p for p in root.rglob("images") if p.is_dir()):
        labels_dir = images_dir.parent / "labels"
        if not labels_dir.is_dir():
            continue
        for image_path in sorted(images_dir.rglob("*")):
            if image_path.suffix.lower() not in IMAGE_EXTS or image_path in seen:
                continue
            seen.add(image_path)
            rel = image_path.relative_to(images_dir)
            label_path = labels_dir / rel.with_suffix(".txt")
            source_split = (rel.parts[0] if len(rel.parts) > 1 else images_dir.parent.name).lower()
            yield image_path, (label_path if label_path.exists() else None), source_split
    if seen:
        return
    for image_path in sorted(root.rglob("*")):
        if image_path.suffix.lower() not in IMAGE_EXTS:
            continue
        label_path = image_path.with_suffix(".txt")
        if label_path.exists():
            yield image_path, label_path, image_path.parent.name.lower()


def split_for(stem: str, val_frac: float, test_frac: float) -> str:
    """Hash-based split.

    A pure function of the file stem, so re-running this script or adding
    images later never moves an existing image between splits. That is what
    stops train/test leakage from creeping in across reruns.
    """
    digest = hashlib.md5(stem.encode("utf-8")).hexdigest()
    bucket = int(digest[:8], 16) / 0xFFFFFFFF
    if bucket < test_frac:
        return "test"
    if bucket < test_frac + val_frac:
        return "val"
    return "train"


def clean_stem(stem: str) -> str:
    """Roboflow appends `.rf.<hash>` to every file; hashing on the part before
    it keeps the split stable across Roboflow re-exports of the same image."""
    return re.split(r"\.rf\.[0-9a-f]+$", stem)[0]


def convert_label(label_path, source_names, dropped: Counter):
    """Source YOLO lines -> lines in this repo's class order. Coordinates are
    clipped to [0, 1]; degenerate boxes and unknown classes are dropped."""
    lines = []
    if label_path is None:
        return lines
    for raw in label_path.read_text(encoding="utf-8", errors="replace").splitlines():
        parts = raw.split()
        if len(parts) < 5:
            continue
        try:
            src_id = int(float(parts[0]))
            cx, cy, bw, bh = (float(v) for v in parts[1:5])
        except ValueError:
            continue
        if source_names is not None and 0 <= src_id < len(source_names):
            raw_name = source_names[src_id]
        else:
            raw_name = CLASSES[src_id] if 0 <= src_id < len(CLASSES) else str(src_id)
        name = ALIASES.get(raw_name.strip().lower())
        if name is None:
            dropped[raw_name] += 1
            continue
        x1, y1 = max(0.0, cx - bw / 2), max(0.0, cy - bh / 2)
        x2, y2 = min(1.0, cx + bw / 2), min(1.0, cy + bh / 2)
        if x2 - x1 <= 0.001 or y2 - y1 <= 0.001:
            dropped["degenerate box"] += 1
            continue
        lines.append("{} {:.6f} {:.6f} {:.6f} {:.6f}".format(
            CLASS_TO_ID[name], (x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1))
    return lines


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True, help="downloaded dataset root (any depth)")
    ap.add_argument("--out", required=True, help="output directory for the YOLO dataset")
    ap.add_argument("--val-frac", type=float, default=0.15)
    ap.add_argument("--test-frac", type=float, default=0.15)
    ap.add_argument("--copy", action="store_true", help="copy images instead of symlinking")
    args = ap.parse_args()

    root = Path(args.root)
    out = Path(args.out)
    if not root.is_dir():
        raise SystemExit(f"{root} is not a directory")

    source_names = read_source_names(root)
    print("source class order:", source_names or "(no yaml found; assuming this repo's order)")

    for split in ("train", "val", "test"):
        (out / "images" / split).mkdir(parents=True, exist_ok=True)
        (out / "labels" / split).mkdir(parents=True, exist_ok=True)

    per_split = Counter()
    per_split_class = {s: Counter() for s in ("train", "val", "test")}
    source_split_counts = Counter()
    dropped = Counter()
    content_hashes = {}
    duplicates = 0
    background = 0
    total = 0

    pairs = list(find_pairs(root))
    if not pairs:
        raise SystemExit(f"no images/ + labels/ folder pairs found under {root}")

    for image_path, label_path, source_split in pairs:
        data = image_path.read_bytes()
        digest = hashlib.md5(data).hexdigest()
        if digest in content_hashes:
            duplicates += 1  # same bytes already kept under another name
            continue
        content_hashes[digest] = image_path.name

        stem = clean_stem(image_path.stem)
        split = split_for(stem, args.val_frac, args.test_frac)
        lines = convert_label(label_path, source_names, dropped)
        if not lines:
            background += 1  # kept: an empty tank teaches the model restraint

        dst_name = stem + image_path.suffix.lower()
        dst_img = out / "images" / split / dst_name
        if not dst_img.exists():
            if args.copy:
                shutil.copy2(image_path, dst_img)
            else:
                try:
                    dst_img.symlink_to(image_path.resolve())
                except OSError:
                    shutil.copy2(image_path, dst_img)  # Windows without developer mode
        (out / "labels" / split / (stem + ".txt")).write_text("\n".join(lines), encoding="utf-8")

        total += 1
        per_split[split] += 1
        source_split_counts[f"{source_split}->{split}"] += 1
        for line in lines:
            per_split_class[split][CLASSES[int(line.split()[0])]] += 1

    (out / "data.yaml").write_text(
        "\n".join([
            f"path: {out.resolve().as_posix()}",
            "train: images/train",
            "val: images/val",
            "test: images/test",
            "names:",
            *[f"  {i}: {n}" for i, n in enumerate(CLASSES)],
            "",
        ]),
        encoding="utf-8",
    )

    stats = {
        "source": "kaggle:slavkoprytula/aquarium-data-cots (Roboflow Aquarium Combined, CC BY 4.0)",
        "classes": CLASSES,
        "images_total": total,
        "images_per_split": dict(per_split),
        "instances_per_split": {s: dict(c) for s, c in per_split_class.items()},
        "instances_total": dict(sum(per_split_class.values(), Counter())),
        "background_only_images": background,
        "byte_identical_duplicates_removed": duplicates,
        "labels_dropped": dict(dropped),
        "source_split_to_our_split": dict(source_split_counts),
        "split_method": "md5(file stem without Roboflow's .rf.<hash> suffix) -> deterministic bucket, image level",
        "val_frac": args.val_frac,
        "test_frac": args.test_frac,
    }
    (out / "split_stats.json").write_text(json.dumps(stats, indent=2), encoding="utf-8")
    print(json.dumps(stats, indent=2))
    print(f"\nwrote {out / 'data.yaml'}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/repo/scripts/train.py
"""Fine-tune RT-DETR on the aquarium dataset.

Sized for the Kaggle free tier. RT-DETR is attention heavy, so batch 8 per
GPU at 640 px is the largest that reliably fits on a 16 GB card. With two
GPUs pass --device 0,1 and --batch 16; Ultralytics trains with DDP across
both. The dataset is small (~450 training images), so the budget is spent on
epochs rather than on data: 80 epochs is about 40 minutes on 2 x T4.

Usage:
    python scripts/train.py --data /kaggle/working/data/aquarium/data.yaml \
        --epochs 80 --batch 16 --device 0,1 --cache ram --project /kaggle/working/runs

Dry run first to measure seconds per epoch before committing the full budget:
    python scripts/train.py --data ... --epochs 2 --name dryrun
"""
import argparse
import json
import platform
import subprocess
import time
import traceback
from pathlib import Path


def hardware_report():
    info = {
        "python": platform.python_version(),
        "platform": platform.platform(),
    }
    try:
        import torch
        info["torch"] = torch.__version__
        info["cuda_available"] = torch.cuda.is_available()
        if torch.cuda.is_available():
            info["gpu_count"] = torch.cuda.device_count()
            info["gpu"] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
            info["gpu_memory_gb"] = round(
                torch.cuda.get_device_properties(0).total_memory / 1024 ** 3, 1)
    except Exception as exc:  # pragma: no cover
        info["torch_error"] = str(exc)
    try:
        info["nvidia_smi"] = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
             "--format=csv,noheader"], text=True).strip()
    except Exception:
        pass
    return info


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--model", default="rtdetr-l.pt")
    ap.add_argument("--epochs", type=int, default=80)
    ap.add_argument("--batch", type=int, default=8)
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--lr0", type=float, default=1e-4)
    ap.add_argument("--optimizer", default="AdamW")
    ap.add_argument("--patience", type=int, default=25)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--device", default="0",
                    help="cuda device id(s), e.g. 0 or 0,1 for multi-GPU DDP, or cpu")
    ap.add_argument("--cache", default="",
                    help="'ram' or 'disk' to cache decoded images; empty for none")
    ap.add_argument("--project", default="runs")
    ap.add_argument("--name", default="rtdetr_aquarium")
    args = ap.parse_args()

    from ultralytics import RTDETR, settings

    # Ultralytics auto-registers a Ray Tune callback whenever `ray` is importable.
    # Kaggle ships a newer ray whose private API that callback calls no longer
    # exists, which crashes training at the end of the first epoch.
    settings.update({"raytune": False})

    hw = hardware_report()
    print(json.dumps(hw, indent=2))

    model = RTDETR(args.model)
    run_dir = Path(args.project) / args.name
    started = time.time()
    error = None
    try:
        model.train(
            data=args.data,
            epochs=args.epochs,
            batch=args.batch,
            imgsz=args.imgsz,
            lr0=args.lr0,
            optimizer=args.optimizer,
            patience=args.patience,
            workers=args.workers,
            seed=args.seed,
            device=args.device,
            project=args.project,
            name=args.name,
            exist_ok=True,
            amp=True,          # required to fit batch 8 on 16 GB
            cache=args.cache or False,
            plots=True,
            val=True,
            # Underwater frames are dim and blue-shifted; hue jitter would make
            # colour a less useful cue, so it stays at the Ultralytics default
            # (0.015). Flips are safe: no animal here has a canonical orientation.
            fliplr=0.5,
        )
    except BaseException as exc:
        # Under DDP the real training runs in subprocesses that already saved
        # best.pt; a crash in the parent must not lose the receipt.
        error = f"{type(exc).__name__}: {exc}"
        traceback.print_exc()
    elapsed = time.time() - started

    epochs_completed = None
    results_csv = run_dir / "results.csv"
    if results_csv.exists():
        epochs_completed = max(0, len(results_csv.read_text(encoding="utf-8").strip().splitlines()) - 1)

    receipt = {
        "status": "failed" if error else "ok",
        "error": error,
        "epochs_completed": epochs_completed,
        "hardware": hw,
        "model": args.model,
        "epochs": args.epochs,
        "batch": args.batch,
        "imgsz": args.imgsz,
        "lr0": args.lr0,
        "optimizer": args.optimizer,
        "patience": args.patience,
        "seed": args.seed,
        "device": args.device,
        "workers": args.workers,
        "cache": args.cache or False,
        "amp": True,
        "data": args.data,
        "wall_clock_seconds": round(elapsed, 1),
        "wall_clock_human": f"{elapsed / 3600:.2f} h",
        "weights": str(run_dir / "weights" / "best.pt"),
    }
    run_dir.mkdir(parents=True, exist_ok=True)
    (run_dir / "training_receipt.json").write_text(json.dumps(receipt, indent=2), encoding="utf-8")
    print(json.dumps(receipt, indent=2))
    print("\nPaste the receipt above into the memo. Reproducibility is graded.")
    if error:
        raise SystemExit(1)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/repo/scripts/evaluate.py
"""Evaluate the fine-tuned model on the validation and the held-out test split.

Both are reported on purpose. The validation number was used for model
selection (best.pt is the best validation epoch), so it is optimistic; the
test split was never looked at during training and is the honest one. The
confusion matrix is exported as numbers, not only as a picture, because
"which class gets mistaken for which" is the question the census reasoning
layer has to defend against.

Usage:
    python scripts/evaluate.py --weights runs/rtdetr_aquarium/weights/best.pt \
        --data /kaggle/working/data/aquarium/data.yaml \
        --out  artifacts/metrics.json
"""
import argparse
import json
from pathlib import Path

from prepare_data import CLASSES


def confusion_as_dict(results):
    """Ultralytics' matrix is (nc+1) x (nc+1): rows are predictions, columns
    are ground truth, and the extra index is `background` (a miss or a false
    positive). Rewritten with class names so it reads without the docs."""
    try:
        matrix = results.confusion_matrix.matrix
    except Exception:
        return None
    labels = list(CLASSES) + ["background"]
    n = min(len(labels), matrix.shape[0])
    table = {}
    for i in range(n):
        row = {}
        for j in range(n):
            v = int(matrix[i, j])
            if v:
                row[labels[j]] = v
        table[labels[i]] = row
    # The two numbers a reader wants first.
    confusions = []
    for i in range(n - 1):
        for j in range(n - 1):
            if i != j and int(matrix[i, j]) > 0:
                confusions.append({"true": labels[j], "predicted": labels[i], "count": int(matrix[i, j])})
    confusions.sort(key=lambda c: -c["count"])
    return {"rows_are_predicted_cols_are_true": table, "off_diagonal_confusions": confusions}


def per_class_metrics(box):
    """Per-class P / R / mAP keyed by class name.

    Ultralytics stores per-class arrays only for classes that have ground
    truth in the split, ordered by `ap_class_index`, so `class_result()` takes
    a position in that list, not a class id. Indexing it by class id silently
    returns another class's numbers whenever one class is absent.
    """
    raw_index = getattr(box, "ap_class_index", None)   # numpy array; never test it with `or`
    ap_index = [int(c) for c in (raw_index if raw_index is not None else [])]
    per_class = {}
    for i, name in enumerate(CLASSES):
        if i not in ap_index:
            per_class[name] = {"note": "no ground-truth instances of this class in the split"}
            continue
        p, r, ap50, ap = box.class_result(ap_index.index(i))
        per_class[name] = {
            "precision": round(float(p), 4),
            "recall": round(float(r), 4),
            "mAP50": round(float(ap50), 4),
            "mAP50_95": round(float(ap), 4),
        }
    return per_class


def run_split(model, data_yaml, split, imgsz, batch, tag, project):
    results = model.val(data=data_yaml, split=split, imgsz=imgsz, batch=batch,
                        plots=True, project=project, name=f"val_{tag}", exist_ok=True)
    box = results.box
    per_class = per_class_metrics(box)
    return {
        "split": split,
        "data": str(data_yaml),
        "mAP50": round(float(box.map50), 4),
        "mAP50_95": round(float(box.map), 4),
        "precision": round(float(box.mp), 4),
        "recall": round(float(box.mr), 4),
        "per_class": per_class,
        "confusion": confusion_as_dict(results),
        "speed_ms_per_image": {k: round(float(v), 2) for k, v in (results.speed or {}).items()},
    }


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--batch", type=int, default=8)
    ap.add_argument("--out", default="artifacts/metrics.json")
    ap.add_argument("--project", default="runs/eval", help="where Ultralytics writes the val plots")
    args = ap.parse_args()

    from ultralytics import RTDETR

    model = RTDETR(args.weights)
    report = {
        "weights": args.weights,
        "classes": CLASSES,
        "validation": run_split(model, args.data, "val", args.imgsz, args.batch, "val", args.project),
        "test": run_split(model, args.data, "test", args.imgsz, args.batch, "test", args.project),
    }
    report["val_minus_test_mAP50"] = round(report["validation"]["mAP50"] - report["test"]["mAP50"], 4)

    out = Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(json.dumps({k: v for k, v in report.items() if k != "validation"}, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/repo/scripts/failure_cases.py
"""Mine the test split for the model's worst images and attach measured
evidence for each failure, so the memo's root-cause analysis is grounded in
numbers rather than in what an annotated image looks like at a glance.

For every test image it matches predictions to ground truth greedily by IoU,
counts false negatives, false positives and class confusions, then measures
five image properties that commonly explain them in aquarium footage:

  blur        variance of the Laplacian over the missed region
  contrast    standard deviation of luminance over the missed region
              (glass, water and backlighting flatten it)
  scale       missed box area as a fraction of image area
  crowding    max IoU between the missed box and any other ground-truth box
              (schools of fish, huddles of penguins)
  exposure    mean luminance of the missed region

Usage:
    python scripts/failure_cases.py --weights runs/rtdetr_aquarium/weights/best.pt \
        --data /kaggle/working/data/aquarium/data.yaml --top 8 \
        --out artifacts/failures
"""
import argparse
import json
from pathlib import Path

import cv2
import numpy as np

from prepare_data import CLASSES

# BGR, one per class, chosen to stay apart on blue-green water.
COLOURS = {
    0: (0, 200, 255),    # fish       amber
    1: (255, 105, 180),  # jellyfish  pink
    2: (255, 255, 255),  # penguin    white
    3: (0, 165, 255),    # puffin     orange
    4: (0, 220, 0),      # shark      green
    5: (0, 0, 255),      # starfish   red
    6: (255, 200, 0),    # stingray   cyan
}


def yolo_to_xyxy(line, width, height):
    cls, cx, cy, bw, bh = (float(v) for v in line.split()[:5])
    x1 = (cx - bw / 2) * width
    y1 = (cy - bh / 2) * height
    x2 = (cx + bw / 2) * width
    y2 = (cy + bh / 2) * height
    return int(cls), np.array([x1, y1, x2, y2], dtype=float)


def iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    if inter <= 0:
        return 0.0
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    return inter / (area_a + area_b - inter)


def crop_stats(image, box):
    h, w = image.shape[:2]
    x1, y1, x2, y2 = [int(max(0, v)) for v in box]
    x2, y2 = min(x2, w), min(y2, h)
    if x2 - x1 < 2 or y2 - y1 < 2:
        return {"blur_laplacian_var": None, "mean_luminance": None, "contrast_std": None}
    crop = image[y1:y2, x1:x2]
    grey = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    return {
        "blur_laplacian_var": round(float(cv2.Laplacian(grey, cv2.CV_64F).var()), 2),
        "mean_luminance": round(float(grey.mean()), 2),
        "contrast_std": round(float(grey.std()), 2),
    }


def diagnose(record):
    """Turn measurements into a first-pass hypothesis. Verify it by eye before
    it goes in the memo. This narrows the search, it does not replace looking."""
    reasons = []
    if record["scale_fraction"] is not None and record["scale_fraction"] < 0.003:
        reasons.append("small object: under 0.3 percent of image area (about 20 px at 640)")
    if record["blur_laplacian_var"] is not None and record["blur_laplacian_var"] < 60:
        reasons.append("motion or focus blur: low Laplacian variance")
    if record["contrast_std"] is not None and record["contrast_std"] < 22:
        reasons.append("low contrast: the animal barely separates from the water")
    if record["max_overlap_with_other_gt"] > 0.30:
        reasons.append("crowding: heavy overlap with a neighbouring box (school or huddle)")
    if record["mean_luminance"] is not None and record["mean_luminance"] < 50:
        reasons.append("underexposed region")
    if record["mean_luminance"] is not None and record["mean_luminance"] > 205:
        reasons.append("blown highlights or backlighting")
    if record["kind"] == "class_confusion":
        reasons.append("class confusion: predicted {} for a {}".format(
            record["predicted_class"], record["true_class"]))
    return reasons or ["no single measured cause stands out, inspect manually"]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", required=True)
    ap.add_argument("--data", required=True)
    ap.add_argument("--conf", type=float, default=0.25)
    ap.add_argument("--iou-match", type=float, default=0.5)
    ap.add_argument("--top", type=int, default=5)
    ap.add_argument("--out", default="artifacts/failures")
    args = ap.parse_args()

    from ultralytics import RTDETR

    data_root = Path(args.data).parent
    img_dir = data_root / "images" / "test"
    lbl_dir = data_root / "labels" / "test"
    images = sorted(p for p in img_dir.iterdir()
                    if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
    if not images:
        raise SystemExit("no test images under {}".format(img_dir))

    model = RTDETR(args.weights)
    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)

    scored = []
    confusion_pairs = {}
    for image_path in images:
        image = cv2.imread(str(image_path))
        if image is None:
            continue
        h, w = image.shape[:2]

        label_path = lbl_dir / (image_path.stem + ".txt")
        gt = []
        if label_path.exists():
            for line in label_path.read_text(encoding="utf-8").splitlines():
                if line.strip():
                    gt.append(yolo_to_xyxy(line, w, h))

        result = model.predict(str(image_path), conf=args.conf, verbose=False)[0]
        preds = []
        for box in result.boxes:
            preds.append((int(box.cls.item()),
                          box.xyxy[0].cpu().numpy().astype(float),
                          float(box.conf.item())))

        matched_pred = set()
        errors = []
        for gi, (gcls, gbox) in enumerate(gt):
            best_j, best_iou = -1, 0.0
            for j, (pcls, pbox, _) in enumerate(preds):
                if j in matched_pred:
                    continue
                score = iou(gbox, pbox)
                if score > best_iou:
                    best_iou, best_j = score, j

            if best_iou < args.iou_match:
                kind, pred_cls = "missed_detection", None
            else:
                matched_pred.add(best_j)
                if preds[best_j][0] == gcls:
                    continue
                kind, pred_cls = "class_confusion", CLASSES[preds[best_j][0]]
                key = "{} -> {}".format(CLASSES[gcls], pred_cls)
                confusion_pairs[key] = confusion_pairs.get(key, 0) + 1

            others = [b for k, (_, b) in enumerate(gt) if k != gi]
            rec = {
                "kind": kind,
                "true_class": CLASSES[gcls],
                "predicted_class": pred_cls,
                "box_xyxy": [round(v, 1) for v in gbox.tolist()],
                "best_iou": round(best_iou, 3),
                "scale_fraction": round(
                    float((gbox[2] - gbox[0]) * (gbox[3] - gbox[1]) / (w * h)), 5),
                "max_overlap_with_other_gt": round(
                    max((iou(gbox, o) for o in others), default=0.0), 3),
            }
            rec.update(crop_stats(image, gbox))
            rec["hypotheses"] = diagnose(rec)
            errors.append(rec)

        false_positives = []
        for j, (pcls, pbox, conf) in enumerate(preds):
            if j in matched_pred:
                continue
            # A false positive that sits on top of a kept prediction of the same
            # class is a duplicate query, which is a different failure from a
            # hallucinated animal; record which one it is.
            twin = max((iou(pbox, preds[k][1]) for k in matched_pred if preds[k][0] == pcls), default=0.0)
            false_positives.append({
                "kind": "false_positive",
                "predicted_class": CLASSES[pcls],
                "confidence": round(conf, 3),
                "box_xyxy": [round(v, 1) for v in pbox.tolist()],
                "iou_with_matched_same_class": round(twin, 3),
                "hypotheses": ["duplicate query on an already-detected animal"] if twin > 0.5
                              else ["unlabelled or hallucinated object, inspect manually"],
            })

        total = len(errors) + len(false_positives)
        if total == 0:
            continue
        scored.append({
            "image": image_path.name,
            "image_size": [w, h],
            "ground_truth_count": len(gt),
            "prediction_count": len(preds),
            "error_count": total,
            "errors": errors,
            "false_positives": false_positives,
        })

    scored.sort(key=lambda r: (-r["error_count"], r["image"]))
    worst = scored[: args.top]

    for record in worst:
        path = img_dir / record["image"]
        image = cv2.imread(str(path))
        scale = max(image.shape[:2]) / 1000.0
        thick = max(1, int(round(2 * scale)))
        fsize = max(0.45, 0.5 * scale)
        result = model.predict(str(path), conf=args.conf, verbose=False)[0]
        for box in result.boxes:
            cls = int(box.cls.item())
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            colour = COLOURS.get(cls, (255, 255, 255))
            cv2.rectangle(image, (x1, y1), (x2, y2), colour, thick)
            cv2.putText(image, "{} {:.2f}".format(CLASSES[cls], box.conf.item()),
                        (x1, max(14, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, fsize, colour, thick)
        for err in record["errors"]:
            x1, y1, x2, y2 = [int(v) for v in err["box_xyxy"]]
            cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), max(1, thick - 1))
            cv2.putText(image, "MISS " + err["true_class"],
                        (x1, min(image.shape[0] - 4, y2 + 14)),
                        cv2.FONT_HERSHEY_SIMPLEX, fsize, (0, 0, 255), thick)
        cv2.imwrite(str(out_dir / ("failure_" + record["image"])), image)

    (out_dir / "failure_report.json").write_text(
        json.dumps({"conf": args.conf,
                    "iou_match": args.iou_match,
                    "images_evaluated": len(images),
                    "images_with_errors": len(scored),
                    "class_confusions_across_test_split": dict(
                        sorted(confusion_pairs.items(), key=lambda kv: -kv[1])),
                    "worst": worst}, indent=2),
        encoding="utf-8")

    print(json.dumps([{"image": r["image"], "errors": r["error_count"]} for r in worst],
                     indent=2))
    print("class confusions across the test split:", confusion_pairs)
    print("\nannotated images and the full report are in {}".format(out_dir))


if __name__ == "__main__":
    main()

In [ ]:
from pathlib import Path
Path("/kaggle/working/repo/app/__init__.py").parent.mkdir(parents=True, exist_ok=True)
Path("/kaggle/working/repo/app/__init__.py").touch()
print("created empty", "/kaggle/working/repo/app/__init__.py")

In [ ]:
%%writefile /kaggle/working/repo/app/detector.py
"""RT-DETR inference wrapper.

Loaded once at process start. Everything downstream consumes the plain
dictionaries this module returns, never the Ultralytics result objects, so the
reasoning layer stays testable without a GPU or a checkpoint.
"""
from __future__ import annotations

import io
import logging
import os
import threading
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List

import numpy as np
from PIL import Image

log = logging.getLogger("aquarium.detector")

CLASSES = ["fish", "jellyfish", "penguin", "puffin", "shark", "starfish", "stingray"]
DEFAULT_WEIGHTS = os.getenv("MODEL_WEIGHTS", "artifacts/best.pt")
DEFAULT_CONF = float(os.getenv("CONF_THRESHOLD", "0.25"))
DEFAULT_IMGSZ = int(os.getenv("IMGSZ", "640"))
DEVICE = os.getenv("DETECTOR_DEVICE", "")   # "" lets Ultralytics choose; "cpu" pins it
WEIGHTS_URL = os.getenv("WEIGHTS_URL", "https://github.com/KanagavelAK/aquarium-rtdetr/releases/download/v1.0/best.pt")
WEIGHTS_KAGGLE_DATASET = os.getenv("WEIGHTS_KAGGLE_DATASET", "")


@dataclass
class Detection:
    label: str
    confidence: float
    box_xyxy: List[float]

    def to_dict(self):
        return asdict(self)


class DetectorError(RuntimeError):
    pass


class Detector:
    """Thin, thread-safe wrapper around a fine-tuned RT-DETR checkpoint."""

    def __init__(self, weights: str = DEFAULT_WEIGHTS, imgsz: int = DEFAULT_IMGSZ):
        self.weights = weights
        self.imgsz = imgsz
        self._lock = threading.Lock()
        self._model = None
        self._names = None

    def load(self):
        if self._model is not None:
            return
        path = Path(self.weights)
        if not path.exists() and (WEIGHTS_URL or WEIGHTS_KAGGLE_DATASET):
            log.info("weights missing at %s, downloading", path)
            try:
                from scripts.download_weights import download
                download(path, WEIGHTS_URL, WEIGHTS_KAGGLE_DATASET)
            except Exception as exc:
                log.warning("weights download failed: %s", exc)
        if not path.exists():
            raise DetectorError(
                "model weights not found at {}. Set MODEL_WEIGHTS to the checkpoint, "
                "or run scripts/download_weights.py.".format(path)
            )
        from ultralytics import RTDETR  # imported lazily, it is slow

        started = time.time()
        self._model = RTDETR(str(path))
        self._names = getattr(self._model, "names", None) or {
            i: n for i, n in enumerate(CLASSES)
        }
        log.info("loaded %s in %.1fs, classes=%s", path, time.time() - started, self._names)

    @property
    def ready(self) -> bool:
        return self._model is not None

    def label_for(self, index: int) -> str:
        if isinstance(self._names, dict):
            return str(self._names.get(index, index))
        try:
            return str(self._names[index])
        except Exception:
            return str(index)

    def predict(self, image_bytes: bytes, conf: float = DEFAULT_CONF):
        """Run detection on raw image bytes.

        Returns (detections, image_width, image_height, inference_ms).
        """
        self.load()
        try:
            image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        except Exception as exc:
            raise DetectorError("could not decode the uploaded file as an image") from exc

        array = np.array(image)[:, :, ::-1]  # PIL RGB -> BGR for ultralytics
        started = time.time()
        with self._lock:  # ultralytics predict is not reentrant
            result = self._model.predict(array, conf=conf, imgsz=self.imgsz, verbose=False,
                                         device=DEVICE or None)[0]
        elapsed_ms = (time.time() - started) * 1000.0

        detections = []
        for box in result.boxes:
            detections.append(Detection(
                label=self.label_for(int(box.cls.item())),
                confidence=round(float(box.conf.item()), 4),
                box_xyxy=[round(float(v), 1) for v in box.xyxy[0].tolist()],
            ))
        detections.sort(key=lambda d: d.confidence, reverse=True)
        return detections, image.width, image.height, round(elapsed_ms, 1)


detector = Detector()

In [ ]:
%%writefile /kaggle/working/repo/app/scene.py
"""Turn a flat list of boxes into the census facts the reasoning layer needs.

The detector answers "where are the animals". A census question asks "how
many", "which is the most common", "are there more X than Y" -- and every one
of those is only as good as the boxes it is summed over. Three things make a
raw box list untrustworthy for counting, and this module measures each:

  1. Duplicate queries. RT-DETR has no NMS. When two decoder queries lock
     onto the same animal the API receives two boxes, usually one strong and
     one weak, almost on top of each other. Counting both is a plain error, so
     same-class boxes with IoU >= DUPLICATE_IOU are collapsed to the stronger
     one and the number collapsed is reported.

  2. Crowding. A school of fish or a huddle of penguins produces boxes that
     overlap their neighbours. Where a box overlaps another box of the same
     class by IoU >= CROWD_IOU the detector may have merged two animals into
     one or split one into two; the count for that class carries a crowding
     score (fraction of its boxes in that state) and the guardrail refuses
     counts when it is high.

  3. Identity conflicts. A shark and a stingray box on the same pixels means
     the detector is not sure what the animal is. Where boxes of different
     classes overlap by IoU >= CONFLICT_IOU the pair is recorded and neither
     class can be counted or ranked with confidence.

Nothing here consults a language model. These are arithmetic facts about the
boxes, computed the same way every time.
"""
from __future__ import annotations

import os
from collections import Counter
from dataclasses import dataclass, field
from typing import Dict, List

DUPLICATE_IOU = 0.70   # same class, this much overlap: one animal, two queries
CROWD_IOU = 0.30       # same class, this much overlap: animals touching or overlapping
CONFLICT_IOU = 0.50    # different classes, this much overlap: one animal, two labels

# Detections weaker than this cannot support an assertion. Shared with the
# guardrail in app/reasoning.py, which imports it from here.
ASSERTION_FLOOR = float(os.getenv("LOW_CONFIDENCE_FLOOR", "0.45"))

PLURALS = {"fish": "fish", "jellyfish": "jellyfish", "starfish": "starfish"}


def plural(label: str, n: int) -> str:
    if n == 1:
        return label
    return PLURALS.get(label, label + "s")


def iou(a, b) -> float:
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    if inter <= 0:
        return 0.0
    area_a = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])
    area_b = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


@dataclass
class ClassFacts:
    label: str
    count: int = 0
    confident: int = 0        # boxes at or above the assertion floor
    tentative: int = 0        # boxes below it
    max_confidence: float = 0.0
    mean_confidence: float = 0.0
    crowded: int = 0          # boxes overlapping another box of the same class
    crowding: float = 0.0     # crowded / count
    conflicts: int = 0        # boxes overlapping a box of a different class
    conflicts_with: List[str] = field(default_factory=list)

    def to_dict(self):
        return {
            "count": self.count,
            "confident": self.confident,
            "tentative": self.tentative,
            "max_confidence": self.max_confidence,
            "mean_confidence": self.mean_confidence,
            "crowded": self.crowded,
            "crowding": self.crowding,
            "conflicts": self.conflicts,
            "conflicts_with": self.conflicts_with,
        }


@dataclass
class SceneFacts:
    image_width: int
    image_height: int
    confidence_floor: float
    assertion_floor: float = ASSERTION_FLOOR
    counts: Dict[str, int] = field(default_factory=dict)
    classes: Dict[str, ClassFacts] = field(default_factory=dict)
    total: int = 0
    kinds: int = 0
    ranking: List[tuple] = field(default_factory=list)   # (label, count), most common first
    duplicates_suppressed: int = 0
    conflicts: List[dict] = field(default_factory=list)
    crowding: float = 0.0
    max_confidence: float = 0.0
    mean_confidence: float = 0.0
    animals: List[dict] = field(default_factory=list)    # boxes kept after duplicate suppression
    detections: List[dict] = field(default_factory=list) # raw detector output

    def facts_for(self, label: str) -> ClassFacts:
        return self.classes.get(label) or ClassFacts(label=label)

    def to_dict(self):
        return {
            "image_size": {"width": self.image_width, "height": self.image_height},
            "confidence_floor": self.confidence_floor,
            "assertion_floor": self.assertion_floor,
            "animals_detected": self.total,
            "kinds_detected": self.kinds,
            "counts": self.counts,
            "ranking": [{"label": k, "count": v} for k, v in self.ranking],
            "duplicates_suppressed": self.duplicates_suppressed,
            "identity_conflicts": self.conflicts,
            "crowding": self.crowding,
            "max_confidence": self.max_confidence,
            "mean_confidence": self.mean_confidence,
            "classes": {k: v.to_dict() for k, v in self.classes.items()},
            "animals": self.animals,
            "detections": self.detections,
        }


def build_scene(detections, image_width: int, image_height: int,
                confidence_floor: float) -> SceneFacts:
    """Group raw detections into census facts."""
    dets = [d.to_dict() if hasattr(d, "to_dict") else dict(d) for d in detections]
    dets_sorted = sorted(dets, key=lambda d: -d["confidence"])

    # 1. Duplicate suppression, strongest box first.
    kept = []
    duplicates = 0
    for d in dets_sorted:
        twin = any(k["label"] == d["label"] and iou(k["box_xyxy"], d["box_xyxy"]) >= DUPLICATE_IOU
                   for k in kept)
        if twin:
            duplicates += 1
            continue
        kept.append({"label": d["label"], "confidence": d["confidence"],
                     "box_xyxy": list(d["box_xyxy"]), "crowded": False, "conflict_with": []})

    # 2. Crowding and 3. identity conflicts, over the kept boxes.
    conflicts = []
    for i, a in enumerate(kept):
        for j in range(i + 1, len(kept)):
            b = kept[j]
            overlap = iou(a["box_xyxy"], b["box_xyxy"])
            if a["label"] == b["label"]:
                if overlap >= CROWD_IOU:
                    a["crowded"] = b["crowded"] = True
            elif overlap >= CONFLICT_IOU:
                a["conflict_with"].append(b["label"])
                b["conflict_with"].append(a["label"])
                conflicts.append({"labels": [a["label"], b["label"]],
                                  "confidences": [a["confidence"], b["confidence"]],
                                  "iou": round(overlap, 3)})

    classes: Dict[str, ClassFacts] = {}
    for k in kept:
        cf = classes.setdefault(k["label"], ClassFacts(label=k["label"]))
        cf.count += 1
        if k["confidence"] >= ASSERTION_FLOOR:
            cf.confident += 1
        else:
            cf.tentative += 1
        cf.max_confidence = max(cf.max_confidence, k["confidence"])
        cf.mean_confidence += k["confidence"]
        if k["crowded"]:
            cf.crowded += 1
        if k["conflict_with"]:
            cf.conflicts += 1
            for other in k["conflict_with"]:
                if other not in cf.conflicts_with:
                    cf.conflicts_with.append(other)
    for cf in classes.values():
        cf.mean_confidence = round(cf.mean_confidence / cf.count, 4)
        cf.crowding = round(cf.crowded / cf.count, 3)
        cf.max_confidence = round(cf.max_confidence, 4)

    counts = {k: v.count for k, v in classes.items()}
    ranking = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))
    confidences = [k["confidence"] for k in kept]

    return SceneFacts(
        image_width=image_width,
        image_height=image_height,
        confidence_floor=confidence_floor,
        counts=counts,
        classes=classes,
        total=len(kept),
        kinds=len(classes),
        ranking=ranking,
        duplicates_suppressed=duplicates,
        conflicts=conflicts,
        crowding=round(sum(1 for k in kept if k["crowded"]) / len(kept), 3) if kept else 0.0,
        max_confidence=round(max(confidences), 4) if confidences else 0.0,
        mean_confidence=round(sum(confidences) / len(confidences), 4) if confidences else 0.0,
        animals=kept,
        detections=dets,
    )

In [ ]:
%%writefile /kaggle/working/repo/app/reasoning.py
"""The reasoning layer: routing, structured reasoning, confidence guardrail.

Hand written on purpose. No agent framework is used or needed -- the whole
control flow is the three functions below, called in order by the API:

    route()      decide whether the detector has to run at all, and what the
                 question is asking for (which animals, count or rank or presence)
    guardrail()  decide, from the census facts, whether an honest answer exists
    compose()    put the facts into plain language

The guardrail runs BEFORE the language model sees anything, and it is
deterministic. A model asked to grade its own confidence will talk itself
into an answer; a rule that says "11 of 14 fish boxes overlap another fish
box, so the count is unreliable" will not.

The language model is optional. With no API key configured the layer answers
from templates over the same structured facts, so the API is fully runnable
offline. Set ANTHROPIC_API_KEY to enable the fluent path.
"""
from __future__ import annotations

import logging
import math
import os
import re
from dataclasses import dataclass, field
from typing import List, Optional

from app.scene import ASSERTION_FLOOR, plural

log = logging.getLogger("aquarium.reasoning")

MODEL_ID = os.getenv("ANTHROPIC_MODEL", "claude-opus-5")
LOW_CONFIDENCE_FLOOR = ASSERTION_FLOOR

# A count is refused when more than this fraction of a class's boxes overlap
# another box of the same class (a school, a huddle, a pile).
CROWDING_MAX = float(os.getenv("CROWDING_MAX", "0.34"))
# A count is refused when fewer than this fraction of a class's boxes clear
# the assertion floor.
CONFIDENT_FRACTION_MIN = float(os.getenv("CONFIDENT_FRACTION_MIN", "0.5"))
# A ranking is refused when the top two counts are closer than this.
RANK_MARGIN_FRACTION = float(os.getenv("RANK_MARGIN_FRACTION", "0.15"))

# ---------------------------------------------------------------------------
# Intent routing
# ---------------------------------------------------------------------------

CLASSES = ["fish", "jellyfish", "penguin", "puffin", "shark", "starfish", "stingray"]

# How people refer to each class. Longer phrases are matched first and blanked
# out, so "sea star" never leaks a stray "star", and "jelly fish" never counts
# as "fish".
SYNONYMS = {
    "starfish": ["starfishes", "starfish", "sea stars", "sea star", "sea-stars", "sea-star",
                 "star fish", "seastars", "seastar"],
    "jellyfish": ["jellyfishes", "jellyfish", "jelly fish", "jellies", "jelly"],
    "stingray": ["stingrays", "stingray", "sting rays", "sting ray", "sting-rays", "sting-ray",
                 "rays", "ray"],
    "fish": ["fishes", "fish"],
    "penguin": ["penguins", "penguin"],
    "puffin": ["puffins", "puffin"],
    "shark": ["sharks", "shark"],
}
CLASS_PATTERNS = [(label, re.compile(r"\b(?:" + "|".join(re.escape(s) for s in syns) + r")\b", re.I))
                  for label, syns in SYNONYMS.items()]

GENERIC_WORDS = {
    "animal", "animals", "creature", "creatures", "species", "kind", "kinds", "type", "types",
    "object", "objects", "thing", "things", "life", "wildlife", "population", "inhabitants",
    "everything", "anything", "something", "critters", "marine", "sea", "underwater",
}
SCENE_WORDS = {"tank", "aquarium", "exhibit", "enclosure", "pool", "water", "display", "habitat"}

IMAGE_REFERENCE = re.compile(
    r"\b(image|picture|photo|photograph|frame|shot|scene|here|visible|shown|see|"
    r"this|that|these|those|there)\b", re.I)

# Questions that are plainly not about this image at all.
NON_VISUAL_PATTERN = re.compile(
    r"\b(who\s+(?:are|is)\s+you|what\s+model|your\s+(?:name|architecture|training)|"
    r"capital\s+of|weather\s+(?:today|tomorrow)|what\s+time\s+is\s+it|tell\s+me\s+a\s+joke|"
    r"how\s+do\s+i\s+(?:train|install|deploy|run)|what\s+is\s+the\s+meaning|"
    r"translate|write\s+(?:me\s+)?(?:a|some)\s+(?:poem|code|essay|story)|"
    r"how\s+does\s+(?:rt-?detr|the\s+model|detection)\s+work|"
    r"what\s+(?:is|are)\s+(?:a|an)\s+(?:fish|jellyfish|penguin|puffin|shark|starfish|stingray)|"
    r"define|definition|"
    r"do\s+(?:fish|sharks|penguins|puffins|jellyfish|starfish|stingrays)\s+(?:sleep|dream|feel\s+pain))\b", re.I)

# Things a visitor may reasonably ask that this detector genuinely cannot
# measure: species-level identity, health, size, age, sex, behaviour, colour,
# water conditions. Refused up front, without running the detector.
OUT_OF_SCOPE_VISUAL = re.compile(
    r"\b(what\s+(?:kind|kinds|type|types|sort|species|breed)\s+of|which\s+species|what\s+species|"
    r"species\s+(?:is|are)\s+(?:this|that|these|those|it|they)|identify|"
    r"clownfish|clown\s+fish|goldfish|tuna|salmon|angelfish|nemo|dory|"
    r"great\s+white|hammerhead|tiger\s+shark|nurse\s+shark|reef\s+shark|whale|dolphin|"
    r"turtle|octopus|seal|otter|crab|lobster|eel|coral|manta|emperor|king\s+penguin|"
    r"healthy|health|sick|ill|injured|hurt|dying|dead|alive|disease|diseased|"
    r"how\s+(?:big|large|small|long|tall|heavy|old|deep|warm|cold)|size|length|weight|weigh|"
    r"age|old|young|male|female|gender|sex|pregnant|baby|babies|juvenile|adult|"
    r"colou?rs?|pattern|stripes?|spotted|"
    r"temperature|salinity|\bph\b|clean|dirty|murky|clear|"
    r"hungry|feeding|eating|fed|food|"
    r"named?|called|happy|sad|mood|feel|feels|"
    r"swimming\s+(?:fast|slowly|towards|away)|direction|speed|"
    r"time\s+of\s+day|brand|logo|text|sign|read)\b", re.I)

COUNT_PATTERN = re.compile(r"\b(how\s+many|count|number\s+of|total|how\s+much|tally|census)\b", re.I)
KINDS_PATTERN = re.compile(r"\b(species|kinds?|types?|sorts?|different|distinct|varieties|variety|diverse|diversity)\b", re.I)
MOST_PATTERN = re.compile(
    r"\b(most\s+common|most\s+frequent|most\s+numerous|most\s+abundant|most\s+of|dominant|dominates?|"
    r"majority|biggest\s+group|largest\s+group|largest\s+number|highest\s+number|"
    r"appears?\s+(?:the\s+)?most|see\s+(?:the\s+)?most|most\s+often|the\s+most|mostly|"
    r"main\s+animal|primary\s+animal)\b", re.I)
LEAST_PATTERN = re.compile(
    r"\b(least\s+common|least\s+frequent|least\s+numerous|least\s+abundant|rarest|fewest|"
    r"smallest\s+group|smallest\s+number|lowest\s+number|minority|the\s+least|least\s+of)\b", re.I)
COMPARISON_PATTERN = re.compile(
    r"\b(more\s+.*?\s+than|fewer\s+.*?\s+than|less\s+.*?\s+than|outnumber|outnumbers|outnumbered|"
    r"compared?\s+(?:to|with)|versus|vs\.?|as\s+many\s+.*?\s+as)\b", re.I)
PRESENCE_PATTERN = re.compile(
    r"\b(is\s+there|are\s+there|any|anyone|anybody|someone|is\s+a|is\s+an|"
    r"does\s+it\s+(?:show|contain|have)|do\s+you\s+see|can\s+you\s+see|can\s+i\s+see|"
    r"contains?|present|visible|spot|is\s+(?:this|that|it)\s+an?|are\s+these)\b", re.I)

KIND_COUNT = "count"
KIND_KINDS = "count_kinds"
KIND_RANKING = "ranking"
KIND_COMPARISON = "comparison"
KIND_PRESENCE = "presence"
KIND_SUMMARY = "summary"
KIND_NOT_IMAGE = "not_about_the_image"
KIND_OUT_OF_SCOPE = "out_of_detector_scope"


@dataclass
class Route:
    needs_detection: bool
    kind: str
    rationale: str
    targets: List[str] = field(default_factory=list)   # classes the question names, in order
    direction: str = ""                                # ranking: "most" or "least"
    decided_by: str = "rules"


def extract_targets(question: str) -> List[str]:
    """Classes named in the question, in the order they appear."""
    text = question.lower()
    found = []
    for label, pattern in CLASS_PATTERNS:
        for match in pattern.finditer(text):
            found.append((match.start(), label))
            # blank the span so a shorter synonym of another class cannot re-match it
            text = text[:match.start()] + " " * (match.end() - match.start()) + text[match.end():]
    found.sort()
    ordered = []
    for _, label in found:
        if label not in ordered:
            ordered.append(label)
    return ordered


def route(question: str) -> Route:
    """Decide whether the detector has to run, and what for.

    Deterministic by design. Routing is a cheap, high-traffic decision with a
    small, closed vocabulary, so a rule set is both faster and easier to defend
    than a model call, and it cannot hallucinate a route.
    """
    q = (question or "").strip()
    if not q:
        return Route(False, KIND_NOT_IMAGE, "the question was empty")

    if NON_VISUAL_PATTERN.search(q):
        return Route(False, KIND_NOT_IMAGE,
                     "the question asks about something other than the contents "
                     "of the image, so running the detector would not inform it")

    targets = extract_targets(q)
    tokens = set(re.findall(r"[a-z-]+", q.lower()))
    mentions_generic = bool(tokens & GENERIC_WORDS)
    mentions_scene = bool(tokens & SCENE_WORDS)
    asks_count = bool(COUNT_PATTERN.search(q))
    asks_kinds = bool(KINDS_PATTERN.search(q))

    # "How many kinds of animal are there?" is answerable: it is a count of the
    # detector's own categories. "How many kinds of fish?" is not: within a
    # category the detector cannot tell species apart.
    if asks_count and asks_kinds:
        if targets:
            return Route(False, KIND_OUT_OF_SCOPE,
                         "the question asks how many varieties of {} there are, and this "
                         "detector recognises {} as one category without telling species "
                         "apart".format(plural(targets[0], 2), plural(targets[0], 2)))
        return Route(True, KIND_KINDS,
                     "the question asks how many different kinds of animal are present, "
                     "which is a count over the detector's categories")

    if OUT_OF_SCOPE_VISUAL.search(q):
        return Route(False, KIND_OUT_OF_SCOPE,
                     "the question is about the image but asks for an attribute this "
                     "detector does not predict (species, health, size, age, sex, colour "
                     "or behaviour), so no amount of detection would answer it")

    if not targets and not mentions_generic and not mentions_scene:
        refers_to_image = bool(IMAGE_REFERENCE.search(q))
        descriptive = bool(re.search(r"\b(what|describe|summari[sz]e|tell\s+me|explain|going\s+on)\b", q, re.I))
        if refers_to_image and descriptive and not asks_count and not PRESENCE_PATTERN.search(q):
            return Route(True, KIND_SUMMARY,
                         "the question asks what the image shows in general")
        if refers_to_image or asks_count or PRESENCE_PATTERN.search(q):
            return Route(False, KIND_OUT_OF_SCOPE,
                         "the question seems to be about the image but names nothing among "
                         "the seven categories this detector predicts, so detection cannot "
                         "answer it")
        return Route(False, KIND_NOT_IMAGE,
                     "the question names nothing this detector can see and does "
                     "not refer to the image")

    if COMPARISON_PATTERN.search(q) and len(targets) >= 2:
        return Route(True, KIND_COMPARISON,
                     "the question compares the number of {} with the number of {}".format(
                         plural(targets[0], 2), plural(targets[1], 2)),
                     targets=targets[:2])
    if MOST_PATTERN.search(q):
        return Route(True, KIND_RANKING,
                     "the question asks which animal is the most common, which needs "
                     "reliable counts for every class present", targets=targets, direction="most")
    if LEAST_PATTERN.search(q):
        return Route(True, KIND_RANKING,
                     "the question asks which animal is the least common, which needs "
                     "reliable counts for every class present", targets=targets, direction="least")
    if asks_count:
        what = ", ".join(plural(t, 2) for t in targets) if targets else "all animals"
        return Route(True, KIND_COUNT,
                     "the question asks for a count of {} in the image".format(what), targets=targets)
    if PRESENCE_PATTERN.search(q) and (targets or mentions_generic):
        what = ", ".join(plural(t, 2) for t in targets) if targets else "any animal"
        return Route(True, KIND_PRESENCE,
                     "the question asks whether {} appear in the image".format(what), targets=targets)
    if targets and not mentions_generic:
        # "Sharks?" or "Tell me about the penguins" -- a bare mention reads as presence.
        return Route(True, KIND_PRESENCE,
                     "the question names {} and asks nothing more specific, so it is "
                     "answered as a presence question".format(", ".join(plural(t, 2) for t in targets)),
                     targets=targets)
    return Route(True, KIND_SUMMARY,
                 "the question is about the contents of the image in general")


# ---------------------------------------------------------------------------
# Confidence guardrail
# ---------------------------------------------------------------------------

@dataclass
class Guard:
    sufficient: bool
    reasons: List[str] = field(default_factory=list)


def _count_problems(cf) -> List[str]:
    """Why a per-class count cannot be trusted. Empty means it can."""
    reasons = []
    if cf.count == 0:
        return reasons
    name = plural(cf.label, 2)
    if cf.crowding > CROWDING_MAX:
        reasons.append("{} of the {} {} boxes overlap another {} box (crowding {:.0%}), so "
                       "individual animals in that group cannot be separated and the count "
                       "may be off in either direction".format(
                           cf.crowded, cf.count, cf.label, cf.label, cf.crowding))
    if cf.confident / cf.count < CONFIDENT_FRACTION_MIN:
        reasons.append("only {} of the {} {} detections scored at or above {:.2f}, so most of "
                       "that count rests on weak evidence".format(
                           cf.confident, cf.count, cf.label, LOW_CONFIDENCE_FLOOR))
    if cf.conflicts:
        reasons.append("{} of the {} {} box{} also carr{} a {} label on the same pixels, so the "
                       "detector is unsure what {} animal{} {}".format(
                           cf.conflicts, cf.count, cf.label, "es" if cf.count != 1 else "",
                           "y" if cf.conflicts != 1 else "ies", " or ".join(cf.conflicts_with),
                           "those" if cf.conflicts != 1 else "that",
                           "s" if cf.conflicts != 1 else "", "are" if cf.conflicts != 1 else "is"))
    return reasons


def guardrail(decision: Route, facts) -> Guard:
    """Decide whether the census facts support an honest answer.

    Runs before the language model and never consults it.
    """
    reasons = []
    kind = decision.kind
    targets = decision.targets

    if not facts.detections:
        reasons.append("the detector returned no animals above the confidence "
                       "floor of {:.2f}".format(facts.confidence_floor))
        return Guard(False, reasons)

    if facts.max_confidence < LOW_CONFIDENCE_FLOOR:
        reasons.append("every detection scored below {:.2f}, which is too weak to "
                       "assert anything about this image".format(LOW_CONFIDENCE_FLOOR))
        return Guard(False, reasons)

    if kind == KIND_COUNT:
        if targets:
            for t in targets:
                reasons += _count_problems(facts.facts_for(t))
        else:
            for cf in facts.classes.values():
                reasons += _count_problems(cf)

    elif kind == KIND_KINDS:
        weak = [cf.label for cf in facts.classes.values() if cf.confident == 0]
        if weak:
            reasons.append("{} appear only as detections below {:.2f}, so it is not clear "
                           "whether {} really present".format(
                               ", ".join(plural(w, 2) for w in weak), LOW_CONFIDENCE_FLOOR,
                               "they are" if len(weak) > 1 else "it is"))
        if facts.conflicts:
            reasons.append("{} animal{} carry two different labels on the same pixels, so "
                           "the number of distinct kinds is uncertain".format(
                               len(facts.conflicts), "s" if len(facts.conflicts) != 1 else ""))

    elif kind == KIND_RANKING:
        ranking = facts.ranking
        if decision.direction == "least":
            ranking = list(reversed(ranking))
        if len(ranking) >= 2:
            (a, na), (b, nb) = ranking[0], ranking[1]
            margin = max(1, math.ceil(RANK_MARGIN_FRACTION * max(na, nb)))
            if abs(na - nb) < margin:
                reasons.append("{} ({}) and {} ({}) are too close to rank with confidence; "
                               "a single missed or duplicated box would change the answer".format(
                                   plural(a, 2), na, plural(b, 2), nb))
            reasons += _count_problems(facts.facts_for(a))
            # The runner-up only matters if an error in its count could flip
            # the order, i.e. when it would need less than doubling to catch up.
            if na < 2 * nb:
                reasons += _count_problems(facts.facts_for(b))
        else:
            reasons += _count_problems(facts.facts_for(ranking[0][0]))

    elif kind == KIND_COMPARISON:
        for t in targets[:2]:
            reasons += _count_problems(facts.facts_for(t))

    elif kind == KIND_PRESENCE:
        for t in targets:
            cf = facts.facts_for(t)
            if cf.count and cf.max_confidence < LOW_CONFIDENCE_FLOOR:
                reasons.append("{} {} detected, but every one scored below {:.2f}, which is "
                               "too weak to confirm".format(
                                   cf.count, plural(t, cf.count), LOW_CONFIDENCE_FLOOR))
            if cf.conflicts:
                reasons.append("the {} candidate{} also labelled as {} on the same pixels, "
                               "so the detector is unsure what it is".format(
                                   t, "s are" if cf.conflicts != 1 else " is",
                                   " or ".join(cf.conflicts_with)))

    return Guard(not reasons, reasons)


# ---------------------------------------------------------------------------
# Answer composition
# ---------------------------------------------------------------------------

def _n(label, n):
    return "{} {}".format(n, plural(label, n))


def _list(parts):
    if not parts:
        return ""
    if len(parts) == 1:
        return parts[0]
    return ", ".join(parts[:-1]) + " and " + parts[-1]


def _deterministic_answer(question: str, decision: Route, facts) -> str:
    kind, targets = decision.kind, decision.targets
    ranking = facts.ranking

    if kind == KIND_COUNT:
        if targets:
            parts = []
            for t in targets:
                cf = facts.facts_for(t)
                parts.append(_n(t, cf.count) if cf.count else "no {}".format(plural(t, 2)))
            text = "I count {} in this image.".format(_list(parts))
            if len(targets) == 1 and not facts.facts_for(targets[0]).count:
                text = "I do not detect any {} in this image above the {:.2f} threshold.".format(
                    plural(targets[0], 2), facts.confidence_floor)
        else:
            text = "I count {} in this image: {}.".format(
                _n("animal", facts.total), _list([_n(k, v) for k, v in ranking]))
        if facts.duplicates_suppressed:
            text += " ({} overlapping duplicate box{} collapsed before counting.)".format(
                facts.duplicates_suppressed, "es" if facts.duplicates_suppressed != 1 else "")
        return text

    if kind == KIND_KINDS:
        return "I can distinguish {} kind{} of animal here: {}.".format(
            facts.kinds, "s" if facts.kinds != 1 else "",
            _list(["{} ({})".format(k, v) for k, v in ranking]))

    if kind == KIND_RANKING:
        if decision.direction == "least":
            label, n = ranking[-1]
            text = "The least common animal detected is the {}, with {} of the {} animals".format(
                label, n, facts.total)
            if len(ranking) > 1:
                text += "; the most common is the {} with {}".format(ranking[0][0], ranking[0][1])
            return text + "."
        label, n = ranking[0]
        text = "The most common animal is the {}: {} of the {} animals detected".format(
            label, n, facts.total)
        if len(ranking) > 1:
            text += ", ahead of {}".format(_n(ranking[1][0], ranking[1][1]))
        return text + "."

    if kind == KIND_COMPARISON:
        a, b = targets[0], targets[1]
        na, nb = facts.facts_for(a).count, facts.facts_for(b).count
        if na == nb:
            return "They are equal: I count {} and {}.".format(_n(a, na), _n(b, nb))
        hi, lo = (a, b) if na > nb else (b, a)
        return "There are more {} than {}: {} versus {}.".format(
            plural(hi, 2), plural(lo, 2), _n(hi, max(na, nb)), _n(lo, min(na, nb)))

    if kind == KIND_PRESENCE:
        if not targets:
            return "Yes. I detect {}: {}.".format(
                _n("animal", facts.total), _list([_n(k, v) for k, v in ranking]))
        yes = [t for t in targets if facts.facts_for(t).count]
        no = [t for t in targets if not facts.facts_for(t).count]
        parts = []
        if yes:
            parts.append("Yes, I detect {} (highest confidence {:.2f})".format(
                _list([_n(t, facts.facts_for(t).count) for t in yes]),
                max(facts.facts_for(t).max_confidence for t in yes)))
        if no:
            parts.append("{} do not detect any {} above the {:.2f} threshold".format(
                "I" if not yes else "but I", _list([plural(t, 2) for t in no]), facts.confidence_floor))
        return ". ".join(p if i == 0 else p[0].upper() + p[1:] for i, p in enumerate(parts)) + "."

    # summary
    text = "This looks like an aquarium scene with {} across {} kind{}: {}.".format(
        _n("animal", facts.total), facts.kinds, "s" if facts.kinds != 1 else "",
        _list([_n(k, v) for k, v in ranking]))
    if len(ranking) > 1:
        text += " The most common is the {}.".format(ranking[0][0])
    return text


SYSTEM_PROMPT = """You answer questions about a photograph taken at a public aquarium.

You cannot see the photograph. You are given the structured output of an
RT-DETR object detector fine-tuned on seven classes: fish, jellyfish, penguin,
puffin, shark, starfish and stingray. A separate deterministic step has already
collapsed duplicate boxes, measured crowding and label conflicts, and decided
that these facts are sufficient to answer the question.

Rules:
- Answer only from the structured facts given. Never infer animals, species,
  attributes or context that are not in them.
- Be direct and plain. Two or three sentences at most.
- Give the numbers that matter and say what they are counts of.
- Never describe your own confidence as a percentage. State what was detected.
- "fish" is a category, not a species: never name a species."""


def _llm_answer(question: str, decision: Route, facts) -> Optional[str]:
    api_key = os.getenv("ANTHROPIC_API_KEY")
    if not api_key:
        return None
    try:
        import json

        import anthropic
        from pydantic import BaseModel

        class Answer(BaseModel):
            answer: str

        client = anthropic.Anthropic()
        payload = {k: v for k, v in facts.to_dict().items() if k != "detections"}
        response = client.messages.parse(
            model=MODEL_ID,
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            output_config={"effort": "low"},
            messages=[{
                "role": "user",
                "content": ("Question: {}\nQuestion type: {}\nClasses the question names: {}\n\n"
                            "Detector facts:\n{}".format(
                                question, decision.kind, decision.targets or "none",
                                json.dumps(payload, indent=2, sort_keys=True))),
            }],
            output_format=Answer,
        )
        if response.stop_reason == "refusal":
            log.warning("model declined to answer, falling back to templates")
            return None
        return response.parsed_output.answer.strip()
    except Exception as exc:
        log.warning("language model step failed (%s), falling back to templates", exc)
        return None


def compose(question: str, decision: Route, facts) -> tuple:
    """Return (answer_text, source) where source is 'llm' or 'template'."""
    text = _llm_answer(question, decision, facts)
    if text:
        return text, "llm"
    return _deterministic_answer(question, decision, facts), "template"


def insufficient_message(kind: str, reasons: List[str]) -> str:
    lead = "I do not have enough information to answer that confidently."
    if kind == KIND_NOT_IMAGE:
        lead = ("That question is not about the contents of the image, so I did "
                "not run the detector.")
    elif kind == KIND_OUT_OF_SCOPE:
        lead = ("I cannot answer that. This detector recognises seven kinds of aquarium "
                "animal as categories (fish, jellyfish, penguin, puffin, shark, starfish, "
                "stingray); it cannot identify species, or judge health, size, age, sex, "
                "colour or behaviour.")
    if not reasons:
        return lead
    return lead + " " + " ".join(r[0].upper() + r[1:] + "." for r in reasons)

## 4. Dataset

Downloaded by code, no manual attachment. Roboflow's **Aquarium Combined**
(CC BY 4.0): 638 photographs from the Henry Doorly Zoo (Omaha) and the
National Aquarium (Baltimore), 4,821 boxes over seven classes, shipped in
YOLO format. The Kaggle mirror is tried first; two other mirrors are
fallbacks. The tree is printed so the memo can quote exactly what was used.

In [ ]:
import kagglehub
from pathlib import Path

DATASET_ROOT, DATASET_SLUG = None, None
for slug in DATASET_CANDIDATES:
    try:
        DATASET_ROOT = Path(kagglehub.dataset_download(slug))
        DATASET_SLUG = slug
        break
    except Exception as exc:
        print(f"{slug}: {type(exc).__name__}: {str(exc)[:120]}")
assert DATASET_ROOT is not None, "no dataset mirror could be downloaded; check Internet is ON"
print("using", DATASET_SLUG, "->", DATASET_ROOT)

def tree(path, depth=0, max_depth=3):
    if depth > max_depth: return
    entries = sorted(path.iterdir(), key=lambda p: (not p.is_dir(), p.name))
    for p in entries[:12]:
        if p.is_dir():
            n_img = sum(1 for f in p.rglob("*") if f.suffix.lower() in (".jpg", ".jpeg", ".png"))
            print("  " * depth + f"{p.name}/  ({n_img} images below)")
            tree(p, depth + 1, max_depth)
        else:
            print("  " * depth + f"{p.name}  ({p.stat().st_size/1e3:.0f} KB)")
    if len(entries) > 12:
        print("  " * depth + f"... {len(entries) - 12} more")
tree(DATASET_ROOT)
for y in list(DATASET_ROOT.rglob("*.yaml"))[:2]:
    print(f"\n--- {y.relative_to(DATASET_ROOT)} ---\n{y.read_text()[:600]}")

## 5. Prepare the training data

Walks the download for images with labels, removes byte-identical
duplicates, maps class ids to this repo's fixed order **by name**, and
re-splits with a deterministic `md5(file stem)` bucket (70 / 15 / 15 at the
image level) so reruns never leak training images into the test split. The
class histogram shows the imbalance the memo has to talk about: fish are more
than half of all boxes, starfish under three percent.

In [ ]:
!python scripts/prepare_data.py --root "{DATASET_ROOT}" --out "{DATA}" --val-frac 0.15 --test-frac 0.15

In [ ]:
import matplotlib.pyplot as plt
stats = json.load(open(f"{DATA}/split_stats.json"))
classes = stats["classes"]
fig, ax = plt.subplots(figsize=(9, 3.2))
bottom = [0] * len(classes)
for split, colour in (("train", "#2a7f9e"), ("val", "#f0a04b"), ("test", "#7a5195")):
    vals = [stats["instances_per_split"][split].get(c, 0) for c in classes]
    ax.bar(classes, vals, bottom=bottom, label=split, color=colour)
    bottom = [b + v for b, v in zip(bottom, vals)]
for i, total in enumerate(bottom):
    ax.text(i, total + 15, str(total), ha="center", fontsize=9)
ax.set_ylabel("boxes"); ax.set_title("instances per class and split"); ax.legend(frameon=False)
plt.tight_layout(); plt.savefig(f"{ARTS}/class_distribution.png", dpi=130); plt.show()
print("images per split:", stats["images_per_split"], "| duplicates removed:", stats["byte_identical_duplicates_removed"])

## 6. Train

`rtdetr-l.pt` (COCO-pretrained) is fetched automatically by Ultralytics on
first use. AMP is on (required to fit batch 8), decoded images are cached in
RAM. A `training_receipt.json` with hardware, hyperparameters and wall-clock
time is written next to the weights.

In [ ]:
import shutil
if Path(RUNS, RUN_NAME).exists():
    shutil.rmtree(Path(RUNS, RUN_NAME))   # a leftover run must never be mistaken for this one
    print("removed stale run directory", Path(RUNS, RUN_NAME))

!python scripts/train.py \
    --data "{DATA}/data.yaml" \
    --model rtdetr-l.pt \
    --epochs {RUN_EPOCHS} --batch {BATCH} --imgsz {IMGSZ} --lr0 {LR0} --optimizer AdamW \
    --patience {PATIENCE} --workers {WORKERS} --seed {SEED} --device {DEVICE} --cache {CACHE} \
    --project "{RUNS}" --name {RUN_NAME}

In [ ]:
assert Path(BEST).exists(), f"training did not produce {BEST}"
receipt_path = Path(RUNS) / RUN_NAME / "training_receipt.json"
if not receipt_path.exists():
    # train.py did not reach its receipt step (e.g. the DDP parent crashed after
    # the workers had already saved best.pt). Rebuild it from Ultralytics' own files.
    import csv, yaml
    run_dir = Path(RUNS) / RUN_NAME
    rows = list(csv.DictReader(open(run_dir / "results.csv"))) if (run_dir / "results.csv").exists() else []
    train_args = yaml.safe_load(open(run_dir / "args.yaml")) if (run_dir / "args.yaml").exists() else {}
    last = {k.strip(): v.strip() for k, v in rows[-1].items()} if rows else {}
    receipt = {
        "status": "reconstructed",
        "note": "train.py exited before writing the receipt; values below come from results.csv and args.yaml",
        "hardware": {"gpu": [torch.cuda.get_device_name(i) for i in range(N_GPU)], "torch": torch.__version__},
        "model": train_args.get("model"), "epochs": train_args.get("epochs"), "epochs_completed": len(rows),
        "batch": train_args.get("batch"), "imgsz": train_args.get("imgsz"), "lr0": train_args.get("lr0"),
        "optimizer": train_args.get("optimizer"), "seed": train_args.get("seed"), "device": str(train_args.get("device")),
        "wall_clock_seconds": float(last.get("time", 0)) or None,
        "final_val_mAP50": last.get("metrics/mAP50(B)"), "final_val_mAP50_95": last.get("metrics/mAP50-95(B)"),
        "weights": BEST,
    }
    receipt_path.write_text(json.dumps(receipt, indent=2))
    print("WARNING: receipt was reconstructed; check the end of the training cell output for the error.\n")
print(receipt_path.read_text())
if DRY_RUN:
    print("\nDRY_RUN is on. Read seconds/epoch above, set DRY_RUN = False and EPOCHS, then Save & Run All.")

In [ ]:
# Training curves: box / cls / giou losses and validation mAP per epoch.
import pandas as pd
res = pd.read_csv(f"{RUNS}/{RUN_NAME}/results.csv"); res.columns = [c.strip() for c in res.columns]
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
for ax, cols, title in ((axes[0], ["train/giou_loss", "val/giou_loss"], "box (GIoU) loss"),
                        (axes[1], ["train/cls_loss", "val/cls_loss"], "classification loss"),
                        (axes[2], ["metrics/mAP50(B)", "metrics/mAP50-95(B)"], "validation mAP")):
    for c in cols:
        if c in res: ax.plot(res["epoch"], res[c], label=c.split("/")[-1])
    ax.set_title(title); ax.set_xlabel("epoch"); ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(f"{ARTS}/training_curves.png", dpi=130); plt.show()
best_epoch = int(res["metrics/mAP50(B)"].idxmax())
print(f"best validation mAP50 {res['metrics/mAP50(B)'].max():.3f} at epoch {res.loc[best_epoch, 'epoch']}, "
      f"{len(res)} epochs run")

## 7. Evaluate

mAP50, mAP50-95, precision and recall per class on the **validation** split
(used for model selection, so optimistic) and on the **held-out test** split
(never looked at during training). The confusion matrix is exported as
numbers as well as a picture: which class the model mistakes for which is
what the census guardrail has to defend against.

In [ ]:
!python scripts/evaluate.py \
    --weights "{BEST}" \
    --data "{DATA}/data.yaml" \
    --imgsz {IMGSZ} --batch {BATCH} \
    --out  "{ARTS}/metrics.json" --project "{RUNS}/eval"

In [ ]:
m = json.load(open(f"{ARTS}/metrics.json"))
rows = []
for c in m["classes"]:
    v, t = m["validation"]["per_class"].get(c, {}), m["test"]["per_class"].get(c, {})
    rows.append([c, v.get("mAP50"), t.get("mAP50"), t.get("mAP50_95"), t.get("precision"), t.get("recall")])
rows.append(["**all**", m["validation"]["mAP50"], m["test"]["mAP50"], m["test"]["mAP50_95"], m["test"]["precision"], m["test"]["recall"]])
print("| class | val mAP50 | test mAP50 | test mAP50-95 | test P | test R |\n|---|---|---|---|---|---|")
for r in rows:
    print("| " + " | ".join("—" if x is None else (f"{x:.3f}" if isinstance(x, float) else str(x)) for x in r) + " |")
print("\ntop confusions on the test split:", m["test"]["confusion"]["off_diagonal_confusions"][:6] if m["test"]["confusion"] else "n/a")

from IPython.display import Image as IPImage, display
for name in ("confusion_matrix_normalized.png", "PR_curve.png"):
    p = Path(RUNS) / "eval" / "val_test" / name
    if p.exists(): display(IPImage(filename=str(p), width=560))

## 8. Failure mining

Worst test images by error count, each miss measured for blur, contrast,
scale, crowding and exposure, and each false positive checked for being a
duplicate query on an already-found animal. Confirm every hypothesis by
looking at the annotated image before it goes in the memo.

In [ ]:
!python scripts/failure_cases.py \
    --weights "{BEST}" \
    --data "{DATA}/data.yaml" \
    --top 8 \
    --out "{ARTS}/failures"

In [ ]:
import cv2
report = json.load(open(f"{ARTS}/failures/failure_report.json"))
worst = report["worst"][:5]
fig, axes = plt.subplots(max(len(worst), 1), 1, figsize=(10, 7 * max(len(worst), 1)))
for ax, rec in zip(axes if len(worst) > 1 else [axes], worst):
    img = cv2.cvtColor(cv2.imread(f"{ARTS}/failures/failure_{rec['image']}"), cv2.COLOR_BGR2RGB)
    ax.imshow(img); ax.axis("off")
    hyp = "; ".join(h for e in rec["errors"] for h in e["hypotheses"])
    ax.set_title(f"{rec['image']} — {rec['error_count']} errors (gt {rec['ground_truth_count']}, pred {rec['prediction_count']})\n{hyp[:170]}", fontsize=9)
plt.tight_layout(); plt.show()
print("class confusions across the whole test split:", report["class_confusions_across_test_split"])

## 9. Prediction gallery and README samples

Four test images with predictions drawn, saved as one figure for the README,
plus two individual test images copied into `samples/` so the API examples
work on a clean checkout.

In [ ]:
from ultralytics import RTDETR
model = RTDETR(BEST)
test_images = sorted(Path(f"{DATA}/images/test").glob("*"))
# pick the four test images with the most ground-truth boxes: busier scenes show more
def n_boxes(p):
    lbl = Path(f"{DATA}/labels/test/{p.stem}.txt")
    return len(lbl.read_text().splitlines()) if lbl.exists() else 0
gallery = sorted(test_images, key=lambda p: -n_boxes(p))[:4]
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
for ax, p in zip(axes.ravel(), gallery):
    r = model.predict(str(p), conf=0.25, verbose=False)[0]
    ax.imshow(cv2.cvtColor(r.plot(line_width=2, font_size=10), cv2.COLOR_BGR2RGB)); ax.axis("off")
    counts = {}
    for c in r.boxes.cls.tolist(): counts[model.names[int(c)]] = counts.get(model.names[int(c)], 0) + 1
    ax.set_title(", ".join(f"{v} {k}" for k, v in sorted(counts.items(), key=lambda kv: -kv[1])) or "no detections", fontsize=10)
plt.tight_layout(); plt.savefig(f"{ARTS}/prediction_gallery.png", dpi=110); plt.show()

os.makedirs(f"{ARTS}/samples", exist_ok=True)
for i, img in enumerate(gallery[:2], 1):
    shutil.copy(img, f"{ARTS}/samples/tank_{i:02d}{img.suffix.lower()}")
print("samples:", os.listdir(f"{ARTS}/samples"))

## 10. End-to-end check of the reasoning layer

Runs the same code path as the API's `POST /ask` — route → detect → census
facts → guardrail → compose — on two test images, offline. No API key is
needed; without one the answer comes from templates over the same structured
facts. Watch for at least one honest refusal: a crowded fish school or a
shark/stingray conflict should produce "I do not have enough information".

In [ ]:
os.environ["MODEL_WEIGHTS"] = BEST
sys.path.insert(0, "/kaggle/working/repo")
import importlib
for m in ("app.detector", "app.scene", "app.reasoning"):
    if m in sys.modules: importlib.reload(sys.modules[m])
from app.detector import Detector
from app.scene import build_scene
from app import reasoning

det = Detector(weights=BEST, imgsz=IMGSZ)
QUESTIONS = ["How many fish are in this tank?", "What is the most common animal here?",
             "Are there any sharks?", "Are there more fish than jellyfish?",
             "How many different kinds of animal are there?", "What species of fish is that?",
             "What is the capital of France?"]
demo_log = []
for sample in gallery[:2]:
    image_bytes = sample.read_bytes()
    print("\n=====", sample.name)
    for question in QUESTIONS:
        decision = reasoning.route(question)
        line = {"image": sample.name, "question": question, "kind": decision.kind, "detector_called": decision.needs_detection}
        if not decision.needs_detection:
            line["answer"] = reasoning.insufficient_message(decision.kind, [])
        else:
            dets, w, h, ms = det.predict(image_bytes, conf=0.25)
            facts = build_scene(dets, w, h, 0.25)
            guard = reasoning.guardrail(decision, facts)
            if not guard.sufficient:
                line["answer"] = reasoning.insufficient_message(decision.kind, guard.reasons)
                line["refused"] = True
            else:
                line["answer"], line["source"] = reasoning.compose(question, decision, facts)
            line["evidence"] = {"counts": facts.counts, "crowding": facts.crowding,
                                "duplicates_suppressed": facts.duplicates_suppressed,
                                "conflicts": len(facts.conflicts)}
        demo_log.append(line)
        print(f"\nQ: {question}\n   route: {decision.kind} | detector: {decision.needs_detection}")
        print("   A:", line["answer"])
        if "evidence" in line: print("   evidence:", line["evidence"])
json.dump(demo_log, open(f"{ARTS}/reasoning_demo.json", "w"), indent=2)

## 11. Package the artifacts

Everything the API and the memo need lands in `/kaggle/working/artifacts`,
plus a single zip. Download from the notebook's **Output** tab, then:

- `best.pt` -> `artifacts/best.pt` in the repo (and a GitHub release asset)
- `samples/` -> `samples/` in the repo (the README curl examples use them)
- `metrics.json`, `training_receipt.json`, `split_stats.json`, `failures/`,
  `reasoning_demo.json` -> fill the README results and the memo
- `*.png` -> `artifacts/` and `docs/` for the README figures

then start the API with `uvicorn app.main:app --host 0.0.0.0 --port 8000`.

In [ ]:
shutil.copy(BEST, f"{ARTS}/best.pt")
shutil.copy(f"{RUNS}/{RUN_NAME}/weights/last.pt", f"{ARTS}/last.pt")
shutil.copy(f"{RUNS}/{RUN_NAME}/training_receipt.json", ARTS)
shutil.copy(f"{DATA}/split_stats.json", ARTS)
for plot in ("results.png", "confusion_matrix.png", "confusion_matrix_normalized.png", "PR_curve.png", "labels.jpg"):
    src = Path(RUNS) / RUN_NAME / plot
    if src.exists():
        shutil.copy(src, ARTS)
for plot in ("confusion_matrix_normalized.png", "PR_curve.png"):
    src = Path(RUNS) / "eval" / "val_test" / plot
    if src.exists():
        shutil.copy(src, f"{ARTS}/test_{plot}")

shutil.make_archive(f"{WORK}/aquarium_artifacts", "zip", ARTS)
for p in sorted(Path(ARTS).rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size/1e6:8.1f} MB  {p.relative_to(ARTS)}")
print(f"\nzip: {WORK}/aquarium_artifacts.zip  {Path(f'{WORK}/aquarium_artifacts.zip').stat().st_size/1e6:.1f} MB")